In [1]:
import numpy as np

## Preliminary

If each slot is a 32-bit number, each slab has 8 slots, and address memory natively (as bytes), how many low bits are available for tagging?

In [2]:
np.log2(4 * 8)

5.0

In [3]:
2**5 == 4 * 8 == 32

True

Each address is

In [4]:
0b100000

32

bits past the previous address. Those 5 bits are ours for tagging.

| bit | meaning |
|:----|:--------|
| 1 (lowest) | is integer |
| 2 | is floating point |
| 3 | is enum |
| 4 | is internal |
| 5 | reserved |

With 64-bit numbers, there are 2 potential reserved bits.

## Tagging

In [5]:
class Tagged:
    def __init__(self, data):
        self._data = data

    def __repr__(self):
        return f"<Tagged{''.join(' ' + x if i % 8 == 0 else x for i, x in enumerate(format(self._data, '032b')))}>"

    @property
    def raw(self):
        return self._data

    @property
    def is_int(self):
        return bool(self._data & np.uint32(0b00000000000000000000000000000001))

    @property
    def is_float(self):
        return not bool(self._data & np.uint32(0b00000000000000000000000000000001)) and bool(self._data & np.uint32(0b00000000000000000000000000000010))

    @property
    def is_enum(self):
        return not bool(self._data & np.uint32(0b00000000000000000000000000000011)) and bool(self._data & np.uint32(0b00000000000000000000000000000100))

    @property
    def is_internal(self):
        return not bool(self._data & np.uint32(0b00000000000000000000000000000111)) and bool(self._data & np.uint32(0b00000000000000000000000000001000))

    @property
    def is_external(self):
        return not bool(self._data & np.uint32(0b00000000000000000000000000011111))

    @classmethod
    def from_int(cls, data):
        return cls((np.int32(data).view(np.uint32) << np.uint32(1)) | np.uint32(0b00000000000000000000000000000001))

    @classmethod
    def from_float(cls, data):
        return cls((np.float32(data).view(np.uint32) & np.uint32(0b11111111111111111111111111111110)) | np.uint32(0b00000000000000000000000000000010))

    @classmethod
    def from_enum(cls, data):
        return cls(np.uint32(data) | np.uint32(0b00000000000000000000000000000100))

    @classmethod
    def from_internal(cls, data):
        return cls(np.uint32(data) | np.uint32(0b00000000000000000000000000001000))

    @classmethod
    def from_external(cls, data):
        return cls(np.uint32(data))

    def to_int(self):
        return ((self._data >> np.uint32(1)) | (self._data & np.uint32(0b10000000000000000000000000000000))).view(np.int32)

    def to_float(self):
        return (self._data & np.uint32(0b11111111111111111111111111111100)).view(np.float32)

    def to_enum(self):
        return self._data & np.uint32(0b11111111111111111111111111111000)

    def to_internal(self):
        return self._data & np.uint32(0b11111111111111111111111111110000)

    def to_external(self):
        return self._data

### Test tagging

In [6]:
x = Tagged.from_int(255)
x.is_int, x.is_float, x.is_enum, x.is_internal, x.is_external

(True, False, False, False, False)

In [7]:
print(bin(np.float32(0.111114495).view(np.uint32)), "(lots of 1's at the end)")
x = Tagged.from_float(0.111114495)
x.is_int, x.is_float, x.is_enum, x.is_internal, x.is_external

0b111101111000111000111111111111 (lots of 1's at the end)


(False, True, False, False, False)

In [8]:
x = Tagged.from_enum(8)   # enums are defined in increments of 8
x.is_int, x.is_float, x.is_enum, x.is_internal, x.is_external

(False, False, True, False, False)

In [9]:
x = Tagged.from_internal(32)   # all pointers are in increments of 4 * 8 = 32
x.is_int, x.is_float, x.is_enum, x.is_internal, x.is_external

(False, False, False, True, False)

In [10]:
x = Tagged.from_external(32)   # all pointers are in increments of 4 * 8 = 32
x.is_int, x.is_float, x.is_enum, x.is_internal, x.is_external

(False, False, False, False, True)

In [11]:
Tagged.from_int(0).to_int()

0

In [12]:
Tagged.from_int(1).to_int()

1

In [13]:
Tagged.from_int(-1).to_int()

-1

In [14]:
Tagged.from_int(2**30 - 1).to_int() == 2**30 - 1

True

In [15]:
Tagged.from_int(-2**30).to_int() == -2**30

True

In [16]:
Tagged.from_float(3.14).to_float()

3.1399994

In [17]:
Tagged.from_float(-3.14).to_float()

-3.1399994

In [18]:
Tagged.from_float(np.inf).to_float()

inf

In [19]:
Tagged.from_float(-np.inf).to_float()

-inf

In [20]:
Tagged.from_float(np.nan).to_float()

nan

## Slabs and slab allocator

In [21]:
class Slab:
    def __init__(self, pointer, data):
        self._pointer = pointer
        self._data = data

    @property
    def pointer(self):
        return self._pointer

    def __getitem__(self, i):
        return self._data[i]

    def __setitem__(self, i, x):
        self._data[i] = x

In [22]:
class SlabAllocator:
    def __init__(self, num_slabs):
        self._arena = np.zeros(num_slabs * 4 * 8, np.uint8)
        self._end = np.uint32(len(self._arena))
        self._first_free = self._end
        self._bump_free = np.uint32(0)
        
    def __repr__(self):
        return str(self._arena.view(np.uint32).reshape(-1, 8))
    
    def alloc(self):
        if self._first_free == self._end:
            if self._bump_free == self._end:
                raise RuntimeError("not enough memory")
            else:
                pointer = self._bump_free
                self._bump_free += 4 * 8
        else:
            pointer = self._first_free
            self._first_free = self._arena[self._first_free : self._first_free + 4].view(np.uint32)[0]
        
        return Slab(pointer, self._arena[pointer : pointer + (4 * 8)].view(np.uint32))
    
    def free(self, slab):
        self._arena[slab.pointer : slab.pointer + 4].view(np.uint32)[0] = self._first_free
        self._first_free = slab.pointer

### Test slab allocator

In [23]:
allocator = SlabAllocator(10)
allocator

[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

In [24]:
one = allocator.alloc()
one[:] = 11
allocator

[[11 11 11 11 11 11 11 11]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

In [25]:
two = allocator.alloc()
two[:] = 22
allocator

[[11 11 11 11 11 11 11 11]
 [22 22 22 22 22 22 22 22]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

In [26]:
three = allocator.alloc()
three[:] = 33
allocator

[[11 11 11 11 11 11 11 11]
 [22 22 22 22 22 22 22 22]
 [33 33 33 33 33 33 33 33]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

In [27]:
allocator.free(two)
allocator

[[ 11  11  11  11  11  11  11  11]
 [320  22  22  22  22  22  22  22]
 [ 33  33  33  33  33  33  33  33]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]]

In [28]:
allocator.free(one)
allocator

[[ 32  11  11  11  11  11  11  11]
 [320  22  22  22  22  22  22  22]
 [ 33  33  33  33  33  33  33  33]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]]

In [29]:
four = allocator.alloc()
four[:] = 44
allocator

[[ 44  44  44  44  44  44  44  44]
 [320  22  22  22  22  22  22  22]
 [ 33  33  33  33  33  33  33  33]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]]

In [30]:
five = allocator.alloc()
five[:] = 55
allocator

[[44 44 44 44 44 44 44 44]
 [55 55 55 55 55 55 55 55]
 [33 33 33 33 33 33 33 33]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

In [31]:
six = allocator.alloc()
six[:] = 66
allocator

[[44 44 44 44 44 44 44 44]
 [55 55 55 55 55 55 55 55]
 [33 33 33 33 33 33 33 33]
 [66 66 66 66 66 66 66 66]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

## Types of slabs

In [34]:
def is_list_slab(slab):
    return Tagged(slab[0]).is_int

def is_instance_slab(slab):
    return Tagged(slab[0]).is_external

In [35]:
allocator = SlabAllocator(10)
slab = allocator.alloc()

slab[0] = Tagged.from_int(123).raw

is_list_slab(slab)

True